### Setup

In [1]:
pip install -q deepl python-dotenv pandas

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.1.2 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
import os
from pathlib import Path
from dotenv import load_dotenv

# the notebook runs from its own folder, so walk up to the project root (the folder with .git)
PROJECT_ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / ".git").exists())

# put your key into the .env file in the project root: DEEPL_API_KEY=your-key-here
load_dotenv(PROJECT_ROOT / ".env")

DEEPL_API_KEY = os.getenv("DEEPL_API_KEY")

print("Project root:", PROJECT_ROOT)
print("Key loaded:", bool(DEEPL_API_KEY) and DEEPL_API_KEY != "your-key-here")

Project root: d:\Study\5th_semester\nlp\project\2026_NLP_Course_Project_Detecting_Persuasion_UKRParlTexts
Key loaded: True


In [3]:
import deepl

translator = deepl.Translator(DEEPL_API_KEY)

usage = translator.get_usage()
if usage.character.valid:
    print(f"Characters used: {usage.character.count:,} / {usage.character.limit:,}")

Characters used: 11,484 / 1,000,000


### Data

In [4]:
import pandas as pd

df = pd.read_csv(PROJECT_ROOT / "data" / "original" / "PL_train.csv")

print(df.shape)
print(df.columns)

(2809, 12)
Index(['filename', 'start', 'end', 'content', 'techniques',
       'persuasion_strategies', 'Attack_on_Reputation', 'Justification',
       'Distraction', 'Simplification', 'Call', 'Manipulative_Wording'],
      dtype='object')


In [5]:
first_100 = df.head(100).copy().reset_index(drop=True)

first_100.head()

,filename,start,end,content,techniques,persuasion_strategies,Attack_on_Reputation,Justification,Distraction,Simplification,Call,Manipulative_Wording
0,pl_hate_speech_act_19_12_2024_n08.txt,8047,8101,Ciągle pokutuje mit o Polsce jako kraju tolera...,"['Appeal_to_Hypocrisy', 'Doubt']",['Attack_on_Reputation'],True,False,False,False,False,False
1,pl_defence_22_05_2024_n01.txt,3160,3187,Rzeczywiście to jest hańba.,"['Exaggeration-Minimisation', 'Loaded_Language']",['Manipulative_Wording'],False,False,False,False,False,True
2,pl_current_affairs_09_01_2025_n02.txt,3965,4288,Posłowie Joanna Borowiak i Anna Gembicka z Klu...,['Loaded_Language'],['Manipulative_Wording'],False,False,False,False,False,True
3,pl_eu_03_04_2025_n05.txt,1975,2091,"Zachęcam do tego, żeby na posiedzeniach tej ko...",['Questioning_the_Reputation'],['Attack_on_Reputation'],True,False,False,False,False,False
4,pl_defence_07_03_2025_n08.txt,2405,2485,"Co prawda tam klimatyzacji nie ma, ale przynaj...",[],[],False,False,False,False,False,False


### Test

In [6]:
result = translator.translate_text(
    "Rzeczywiście to jest hańba.",
    source_lang="PL",
    target_lang="UK",
)

print(result.text)

Це справді ганьба.


### Translation

In [7]:
import time

# separate name so the LAPA output (PL_train_first_100_uk.csv) is not overwritten
OUTPUT_FILE = PROJECT_ROOT / "data" / "translated" / "PL_train_first_100_uk_deepl.csv"
BATCH_SIZE = 20  # DeepL accepts up to 50 texts per request

first_100["content_uk"] = ""

for start in range(0, len(first_100), BATCH_SIZE):
    end = min(start + BATCH_SIZE, len(first_100))

    # skip batches that are already translated (safe to re-run)
    if (first_100.loc[start:end - 1, "content_uk"] != "").all():
        continue

    print(f"Translating {start + 1}-{end}/100...")

    texts = first_100.loc[start:end - 1, "content"].astype(str).tolist()

    for attempt in range(3):
        try:
            results = translator.translate_text(
                texts,
                source_lang="PL",
                target_lang="UK",
                preserve_formatting=True,
            )
            break
        except deepl.DeepLException as e:
            print(f"  DeepL error: {e}. Retrying...")
            time.sleep(2 ** attempt)
    else:
        raise RuntimeError(f"Batch {start}-{end} failed after 3 attempts")

    first_100.loc[start:end - 1, "content_uk"] = [r.text for r in results]

    first_100.to_csv(OUTPUT_FILE, index=False)

    for pl, uk in zip(texts, [r.text for r in results]):
        print(uk)
        print("-" * 80)

Translating 1-20/100...
Досі існує міф про Польщу як про толерантну країну.
--------------------------------------------------------------------------------
Це справді ганьба.
--------------------------------------------------------------------------------
Депутати Йоанна Боровіак та Анна Гембіцька з парламентського клубу «Право і справедливість» зададуть запитання щодо рішення генерального директора з питань охорони навколишнього середовища від грудня 2024 року, яке зірвало будівництво гідротехнічної споруди в Сіарзеві — інвестиційного проєкту, забезпеченого урядом партії «Право і Справедливість» та такого, що гарантує захист Куяв від повеней.
--------------------------------------------------------------------------------
Я закликаю, щоб на засіданнях цієї комісії з боку пана депутата було менше лицемірства, а більше співпраці.
--------------------------------------------------------------------------------
Щоправда, там немає кондиціонера, але принаймні солдати всередині не гинуть.


In [8]:
first_100[["content", "content_uk"]].head(10)

,content,content_uk
0,Ciągle pokutuje mit o Polsce jako kraju tolera...,Досі існує міф про Польщу як про толерантну кр...
1,Rzeczywiście to jest hańba.,Це справді ганьба.
2,Posłowie Joanna Borowiak i Anna Gembicka z Klu...,Депутати Йоанна Боровіак та Анна Гембіцька з п...
3,"Zachęcam do tego, żeby na posiedzeniach tej ko...","Я закликаю, щоб на засіданнях цієї комісії з б..."
4,"Co prawda tam klimatyzacji nie ma, ale przynaj...","Щоправда, там немає кондиціонера, але принаймн..."
5,"Drogie, Szanowne Działaczki Ruchów Kobiecych i...","Дорогі, шановні активістки жіночих рухів та ак..."
6,Dzięki liderstwu ministra obrony narodowej Wła...,Завдяки лідерству міністра національної оборон...
7,Dzisiaj jest w pańskiej ręce jedna decyzja jak...,Сьогодні у ваших руках одне рішення як голови ...
8,Witam was serdecznie. (Oklaski) (Poseł Urszula...,Щиро вітаю вас. (Оплески) (Депутатка Уршула Па...
9,W Polsce jedna kobieta na trzy miała aborcję.,У Польщі кожна третя жінка робила аборт.
